# 13 - Evaluate + Test UI: Pest Classifier (YOLOv8)

Stage 13: Evaluate the trained pest classifier (from 11_train_pest_classifier)
and test it interactively via drag-and-drop, same pattern as 04/08/05/10.

Two things in one notebook since they're small and related:
  Part A: proper per-class evaluation on the held-out test split
          (classification_report + confusion matrix -- ultralytics'
          built-in .val() only gives top1/top5 accuracy, not the
          per-class breakdown you've been using elsewhere)
  Part B: Gradio drag-and-drop UI, same style as 05/10

Install deps:
    pip install ultralytics gradio scikit-learn matplotlib seaborn --break-system-packages

## Imports & Configuration

In [ ]:
import sys
from pathlib import Path

import numpy as np
import cv2
import gradio as gr
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from ultralytics import YOLO

sys.path.append(str(Path.cwd()))
from project_config import NOTEBOOKS_DIR  # noqa: E402

PEST_DATASET_DIR = NOTEBOOKS_DIR / "pest_dataset"
RUNS_DIR = NOTEBOOKS_DIR / "../models/pest_classifier"

## `find_latest_weights`

Ultralytics auto-suffixes repeated run names (yolov8n_cls_pest,

In [ ]:
def find_latest_weights():
    """
    Ultralytics auto-suffixes repeated run names (yolov8n_cls_pest,
    yolov8n_cls_pest2, ...) -- this grabs the most recently modified
    best.pt rather than assuming a fixed folder name.
    """
    candidates = list(RUNS_DIR.glob("*/weights/best.pt"))
    if not candidates:
        raise FileNotFoundError(f"No trained weights found under {RUNS_DIR}. Run 11 first.")
    return max(candidates, key=lambda p: p.stat().st_mtime)

## `load_pest_model`

In [ ]:
def load_pest_model():
    weights_path = find_latest_weights()
    print(f"Loading weights from: {weights_path}")
    model = YOLO(str(weights_path))
    return model

## `evaluate_test_set`

In [ ]:
def evaluate_test_set(model):
    test_dir = PEST_DATASET_DIR / "test"
    classes = sorted([d.name for d in test_dir.iterdir() if d.is_dir()])

    y_true, y_pred = [], []
    for class_idx, class_name in enumerate(classes):
        for img_path in (test_dir / class_name).iterdir():
            result = model.predict(str(img_path), verbose=False)[0]
            pred_idx = int(result.probs.top1)
            pred_class = result.names[pred_idx]
            y_true.append(class_name)
            y_pred.append(pred_class)

    print("=== Classification Report ===")
    print(classification_report(y_true, y_pred, digits=3, zero_division=0))
    macro_f1 = f1_score(y_true, y_pred, average="macro")
    print(f"Macro-F1: {macro_f1:.4f}")

    cm_labels = sorted(set(y_true) | set(y_pred))
    cm = confusion_matrix(y_true, y_pred, labels=cm_labels)
    plt.figure(figsize=(max(6, len(cm_labels)), max(5, len(cm_labels) * 0.8)))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=cm_labels, yticklabels=cm_labels)
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.title("Pest Classifier - Confusion Matrix (Test Set)")
    plt.tight_layout()
    plt.savefig("confusion_matrix_pest.png", dpi=150)
    plt.show()
    print("Saved: confusion_matrix_pest.png")

    return macro_f1

## `build_predict_fn`

In [ ]:
def build_predict_fn(model):
    def predict(image_rgb):
        if image_rgb is None:
            return {}
        result = model.predict(image_rgb, verbose=False)[0]
        probs = result.probs.data.cpu().numpy()
        names = result.names
        return {names[i]: float(probs[i]) for i in range(len(probs))}
    return predict

## `launch_ui`

In [ ]:
def launch_ui(model):
    predict_fn = build_predict_fn(model)
    demo = gr.Interface(
        fn=predict_fn,
        inputs=gr.Image(type="numpy", label="Drag & drop a leaf/pest-damage photo here"),
        outputs=gr.Label(num_top_classes=5, label="Pest Prediction"),
        title="Smart Farming - Pest Symptom Classifier (Test UI)",
        description=(
            "Upload or drag & drop a photo showing possible pest damage. "
            "Note: this is a CLASSIFIER (what pest is likely present), not "
            "a detector -- it won't draw a box around the insect itself. "
            "Trained on a small, imbalanced dataset (see the evaluation "
            "above) -- treat low-confidence or borderline predictions with "
            "caution, especially for Thrips if it survived pruning."
        ),
    )
    demo.launch()

## Run

In [ ]:
model = load_pest_model()
macro_f1 = evaluate_test_set(model)
print(f"\nOverall macro-F1: {macro_f1:.4f}")
print("\nLaunching test UI...")
launch_ui(model)